# 🌌 CMB Foreground Separation Workshop
## Monte Carlo / Bayesian Approach

---

## What You'll Learn

In this workshop, you will:
1. Understand how the CMB gets contaminated by our Galaxy
2. Build a mathematical model of what telescopes observe
3. Use Monte Carlo sampling to find the best model parameters
4. Recover a clean CMB map from noisy, contaminated data

**No prior experience with Monte Carlo methods is required!**

---
## Background: The Problem We're Solving

### What is the CMB?

The **Cosmic Microwave Background (CMB)** is the oldest light in the universe. It was released about 380,000 years after the Big Bang, when the universe cooled enough for atoms to form. Before this, light couldn't travel freely — it kept bouncing off charged particles. Once atoms formed, light could finally escape and travel across the universe.

Today, that ancient light has been stretched by the expansion of the universe and appears as microwaves. It's almost perfectly uniform, but contains tiny temperature fluctuations (about 1 part in 100,000) that tell us about the early universe.

---

### The Problem: Galactic Contamination

When we point a telescope at the sky to see the CMB, we don't just see the CMB. We also see **emission from our own Milky Way galaxy** - called "foregrounds" because they're in front of the CMB.

| Component | What Is It? | Physical Origin |
|-----------|-------------|----------------|
| **CMB** | Ancient light from Big Bang | 380,000 years after Big Bang |
| **Synchrotron** | Radio waves | Fast electrons spiraling in magnetic fields |
| **Thermal Dust** | Infrared/microwave glow | Tiny dust grains heated by starlight |

**Our challenge:** Separate the CMB from the foreground contamination!

---

### The Key Insight

Here's the crucial physics that makes separation possible:

| Component | Behavior with Frequency | Why? |
|-----------|------------------------|------|
| **CMB** | **CONSTANT** (same at all frequencies!) | It's a perfect blackbody |
| **Synchrotron** | Gets **less visible** at higher frequencies | Electrons lose energy |
| **Dust** | Gets **more visible** at higher frequencies relative to synchotron | Warm dust emits more at shorter wavelengths |

By observing at multiple frequencies (like 30 GHz, 100 GHz, 353 GHz), we can use these different behaviors to figure out how much of each component is present!

### Our Approach: Bayesian Inference with MCMC

We'll use a technique called **Markov Chain Monte Carlo (MCMC)** to:
1. Build a model that predicts what we should observe
2. Find the model parameters that best match our data
3. Get uncertainties on our estimates (error bars!)

---
## Setup: Loading Libraries

First, we need to import the Python libraries we'll use.

**What each library does:**
- `numpy`: Does math with arrays (lists of numbers)
- `matplotlib`: Makes plots and images
- `tqdm`: Shows progress bars (optional, but nice to have)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

np.random.seed(42) # the meaning of life

print("✓ Setup complete!")

---
## Physical Constants and Frequencies

Here we define the physical constants we need and the frequencies at which we'll "observe" the sky.

**Why these specific frequencies?**

These are the frequencies used by the Planck satellite. They were chosen to:
- Capture synchrotron emission (strong at 30 GHz)
- See the CMB clearly (cleanest around 70-143 GHz)
- Capture dust emission (strong at 353 GHz)

In [ ]:
# Physical constants (you don't need to memorize these!)

# Planck's constant: relates the energy of a photon to its frequency
# E = h × frequency
h_planck = 6.62607015e-34  # Units: Joule × seconds

# Boltzmann's constant: relates temperature to energy
# It tells us how much energy corresponds to a given temperature
k_boltzmann = 1.380649e-23  # Units: Joules per Kelvin

# Speed of light: the fastest anything can travel
c_light = 2.99792458e8  # Units: meters per second (about 300,000 km/s!)

# Observation frequencies (matching Planck satellite)
FREQUENCIES_GHZ = np.array([30, 44, 70, 100, 143, 217, 353])  # In GigaHertz
FREQUENCIES_HZ = FREQUENCIES_GHZ * 1e9  # Convert to Hertz (multiply by 1 billion)

# Reference frequencies for foregrounds
# (We measure synchrotron amplitude at 30 GHz, dust at 353 GHz)
NU_REF_SYNC = 30e9    # 30 GHz in Hz
NU_REF_DUST = 353e9   # 353 GHz in Hz

# Dust temperature (typical value for Milky Way dust)
T_DUST = 20.0  # Kelvin (about -253°C, very cold but warmer than deep space!)

print(f"We'll observe at these frequencies: {FREQUENCIES_GHZ} GHz")

---
# The Physics: Spectral Scaling Functions

This is the **heart of component separation**! Each type of emission has a unique "spectral signature" — how its brightness changes with frequency.

## CMB: Constant at All Frequencies

The CMB is thermal radiation (a "blackbody") at 2.7 Kelvin. When we measure it in temperature units, it looks **exactly the same at all frequencies**.

$$\text{CMB brightness}(\nu) = \text{constant}$$

This is the key property that lets us identify it!

---

## Synchrotron: Power Law (Dimmer at Higher Frequencies)

**What is synchrotron radiation?**
- Electrons in our galaxy are accelerated by magnetic fields
- When charged particles accelerate, they emit light
- The faster they spiral, the more light they emit

**The math:**
$$\text{Synchrotron}(\nu) = A_{\text{sync}} \times \left(\frac{\nu}{30 \text{ GHz}}\right)^{\beta_s}$$

Where:
- $A_{\text{sync}}$ is the amplitude (brightness) at 30 GHz
- $\beta_s \approx -3$ is the "spectral index" (a negative number!)
- $\nu$ is the frequency we're observing at

**Why does β being negative matter?**

When you raise something to a negative power, higher values give smaller results:
- At 30 GHz: $(30/30)^{-3} = 1^{-3} = 1$ (reference point)
- At 100 GHz: $(100/30)^{-3} = (3.33)^{-3} ≈ 0.027$ (much dimmer!)
- At 353 GHz: $(353/30)^{-3} ≈ 0.0006$ (almost zero!)

So synchrotron is **37× dimmer** at 100 GHz than at 30 GHz!

---

## Thermal Dust: Modified Blackbody (Brighter at Higher Frequencies)

**What is thermal dust emission?**
- Tiny dust grains (like smoke particles) float between stars
- They absorb starlight and heat up
- Warm objects glow — the dust re-emits at infrared/microwave wavelengths

**The math:**
$$\text{Dust}(\nu) = A_{\text{dust}} \times \left(\frac{\nu}{353 \text{ GHz}}\right)^{1.5} \times \frac{B_\nu(T_d)}{B_{353}(T_d)}$$

Where $B_\nu(T)$ is the Planck blackbody function — it describes how hot objects emit light.

The key point: dust gets **brighter** at higher frequencies (the opposite of synchrotron!).

In [ ]:
def planck_function(nu, T):
    """
    The Planck blackbody function - describes how hot objects emit light.

    Think of it like this: a hot piece of metal glows red, then orange,
    then white as it gets hotter. This function tells us exactly how
    much light comes out at each frequency for a given temperature.

    Parameters:
        nu: frequency in Hz
        T: temperature in Kelvin

    Returns:
        The brightness at that frequency
    """
    x = h_planck * nu / (k_boltzmann * T)
    return (2 * h_planck * nu**3 / c_light**2) / (np.exp(x) - 1)


def synchrotron_spectrum(nu, beta_s=-3.0):
    """
    How synchrotron emission scales with frequency.

    Synchrotron radiation comes from fast-moving electrons spiraling
    in magnetic fields. It's brightest at low frequencies and gets
    dimmer as frequency increases.

    The formula is a "power law": (frequency ratio)^(power)
    Since beta_s is negative (around -3), higher frequencies give smaller values.

    Parameters:
        nu: frequency in Hz
        beta_s: spectral index (slope of the power law, typically -3)

    Returns:
        Scaling factor (1.0 at 30 GHz, smaller at higher frequencies)
    """
    return (nu / NU_REF_SYNC) ** beta_s


def dust_spectrum(nu):
    """
    How thermal dust emission scales with frequency.

    Dust grains in space absorb starlight and re-emit it at infrared/
    microwave wavelengths. Warmer dust emits more at higher frequencies.

    Parameters:
        nu: frequency in Hz

    Returns:
        Scaling factor (1.0 at 353 GHz, smaller at lower frequencies)
    """
    return (nu / NU_REF_DUST) ** 1.5 * planck_function(nu, T_DUST) / planck_function(NU_REF_DUST, T_DUST)


print("✓ Spectral functions defined!")

### Visualizing the Spectral Signatures

The plot below shows **why component separation works**. Each component has a different "shape" — a unique fingerprint!

- **CMB (black)**: A flat horizontal line — same at all frequencies
- **Synchrotron (blue)**: Slopes downward — dimmer at higher frequencies  
- **Dust (red)**: Slopes upward — brighter at higher frequencies

These different shapes let us mathematically separate the components!

### Visualizing the Observed Maps

Let's look at what the "telescope" observes at different frequencies. Notice how the maps look different at each frequency — this is because the foreground contributions change!

In [ ]:
# Spectral signatures plot
freq_plot = np.logspace(np.log10(10), np.log10(500), 100) * 1e9

plt.figure(figsize=(10, 6))
plt.loglog(freq_plot/1e9, np.ones_like(freq_plot), 'k-', lw=3, label='CMB (constant!)')
plt.loglog(freq_plot/1e9, synchrotron_spectrum(freq_plot)/synchrotron_spectrum(100e9), 'b-', lw=2, label='Synchrotron')
plt.loglog(freq_plot/1e9, dust_spectrum(freq_plot)/dust_spectrum(100e9), 'r-', lw=2, label='Dust')
for f in FREQUENCIES_GHZ:
    plt.axvline(f, color='gray', ls=':', alpha=0.5)
plt.xlabel('Frequency [GHz]')
plt.ylabel('Relative Amplitude')
plt.title('Spectral Signatures of Sky Components')
plt.legend()
plt.xlim(10, 500)
plt.ylim(1e-4, 1e3)
plt.grid(True, alpha=0.3)
plt.show()

## Generating Simulated Sky Data

Now we'll create **fake telescope data** that mimics what a real telescope would see. This lets us:

1. Know the "true answer" (since we created it)
2. Test whether our method can recover that truth

### What We'll Simulate

We'll create 128×128 pixel maps of:
1. **CMB** — temperature fluctuations (this is what we want to recover!)
2. **Synchrotron** — galactic contamination pattern
3. **Dust** — different galactic contamination pattern

Then we'll **combine them** at each frequency using the spectral scaling, and add **noise** (telescopes aren't perfect!).

### The Simulation Model

At each pixel and each frequency, the observation is:

$$\text{observation}(\nu) = \text{CMB} + \text{sync} \times f_s(\nu) + \text{dust} \times f_d(\nu) + \text{noise}$$

Where:
- CMB amplitude is constant (doesn't depend on frequency)
- Synchrotron is multiplied by its scaling function $f_s(\nu)$
- Dust is multiplied by its scaling function $f_d(\nu)$
- Noise is random (Gaussian) with some standard deviation

In [ ]:
# Helper Function
def generate_gaussian_field(npix, power_amplitude=1000, power_slope=-2.5):
    freqs = np.fft.fftfreq(npix)
    fx, fy = np.meshgrid(freqs, freqs)
    k = np.sqrt(fx**2 + fy**2)
    with np.errstate(divide='ignore', invalid='ignore'):
        power = power_amplitude * (k * npix / 10) ** power_slope
        power = np.where(np.isfinite(power) & (k > 0), power, 0)
    phases = np.random.uniform(0, 2*np.pi, (npix, npix))
    fourier = np.sqrt(power / 2) * np.exp(1j * phases)
    return np.real(np.fft.ifft2(fourier)) * npix

In [ ]:
# Generate sky maps
def generate_sky_maps(npix=128, noise_level=30.0, cmb_amp=110.0, sync_amp=70.0, dust_amp=50.0, beta_s=-3.0):
    print("Generating simulated sky maps...")
    print("="*60)
    print("GENERATING SIMULATED SKY DATA")
    print("="*60)
    print(f"Map size: {npix} × {npix} = {npix**2:,} pixels")
    print(f"Frequency channels: {len(FREQUENCIES_GHZ)}")
    print()

    print("Step 1: Generating CMB temperature fluctuations...")
    cmb = generate_gaussian_field(npix, 6000, -0.5)
    cmb = cmb / np.std(cmb) * cmb_amp
    print(f"   CMB RMS amplitude: {np.std(cmb):.1f} μK")

    print("Step 2: Generating synchrotron emission template...")
    sync = np.abs(generate_gaussian_field(npix, 2000, -2.8))
    sync = sync / np.std(sync) * sync_amp
    print(f"   Synchrotron RMS at 30 GHz: {np.std(sync):.1f} μK")

    print("Step 3: Generating thermal dust template...")
    dust = np.abs(generate_gaussian_field(npix, 1500, -2.5))
    dust = dust / np.std(dust) * dust_amp
    print(f"   Dust RMS at 353 GHz: {np.std(dust):.1f} μK")

    print("Step 4: Combining components at each frequency...")
    observed = np.zeros((len(FREQUENCIES_HZ), npix, npix))
    for i, nu in enumerate(FREQUENCIES_HZ):
        observed[i] = cmb + sync * synchrotron_spectrum(nu, beta_s) + dust * dust_spectrum(nu)
        observed[i] += np.random.normal(0, noise_level, (npix, npix))

    print(f"  ✓ Generated {len(FREQUENCIES_GHZ)} maps ({npix}×{npix} pixels)")
    return {'observed_maps': observed, 'cmb_input': cmb, 'sync_input': sync, 'dust_input': dust,
            'frequencies_hz': FREQUENCIES_HZ, 'frequencies_ghz': FREQUENCIES_GHZ,
            'npix': npix, 'noise_std': noise_level, 'beta_s_true': beta_s}

data = generate_sky_maps()

In [ ]:
# Show observations at low, middle, and high frequency
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# Index 0 = 30 GHz, Index 3 = 100 GHz, Index -1 (last) = 353 GHz
indices = [0, 3, -1]
titles = ['30 GHz\n(synchrotron dominates)',
          '100 GHz\n(CMB cleanest here)',
          '353 GHz\n(dust dominates)']

for ax, idx, title in zip(axes, indices, titles):
    img = data['observed_maps'][idx]

    # Set color scale to show the range of values
    vmax = np.percentile(np.abs(img), 98)

    # imshow displays a 2D array as an image
    # cmap='RdBu_r' = red-white-blue colormap (red=hot, blue=cold)
    # origin='lower' = put (0,0) at bottom-left
    im = ax.imshow(img, cmap='RdBu_r', origin='lower', vmin=-vmax, vmax=vmax)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Pixel')
    plt.colorbar(im, ax=ax, label='Temperature [μK]')

axes[0].set_ylabel('Pixel')
plt.suptitle('What the Telescope Observes at Different Frequencies', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("\n📊 Notice how the maps look different!")
print("   • 30 GHz: Extra structure from synchrotron emission")
print("   • 100 GHz: Cleanest view of the CMB")
print("   • 353 GHz: Extra structure from dust emission")

### The True Components (Ground Truth)

In real observations, we wouldn't know the true components. But since we **simulated** the data, we know the answer! This lets us check how well our method works.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# CMB map (what we want to recover!)
vmax = np.percentile(np.abs(data['cmb_input']), 98)
im0 = axes[0].imshow(data['cmb_input'], cmap='RdBu_r', origin='lower', vmin=-vmax, vmax=vmax)
axes[0].set_title('True CMB\n(our goal: recover this!)', fontsize=12)
plt.colorbar(im0, ax=axes[0], label='μK')

# Synchrotron template (contamination #1)
im1 = axes[1].imshow(data['sync_input'], cmap='hot', origin='lower')
axes[1].set_title('Synchrotron Template\n(contamination #1)', fontsize=12)
plt.colorbar(im1, ax=axes[1], label='μK at 30 GHz')

# Dust template (contamination #2)
im2 = axes[2].imshow(data['dust_input'], cmap='hot', origin='lower')
axes[2].set_title('Dust Template\n(contamination #2)', fontsize=12)
plt.colorbar(im2, ax=axes[2], label='μK at 353 GHz')

for ax in axes:
    ax.set_xlabel('Pixel')
axes[0].set_ylabel('Pixel')

plt.suptitle('True Sky Components (Ground Truth)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## Part 1: Building the Model

Now we'll build a mathematical model that predicts what we should observe.

### The Forward Model

A **forward model** takes parameter values and predicts what we would observe. It goes "forward" from parameters to data:

$$\text{Parameters} \xrightarrow{\text{Forward Model}} \text{Predicted Data}$$

We want to solve the **inverse problem**: given data, find the parameters. But first we need the forward model!

For a single pixel, we observe 7 numbers (one per frequency). Our model predicts:

$$\text{model}(\nu) = s_{\text{CMB}} + s_{\text{sync}} \times f_s(\nu) + s_{\text{dust}} \times f_d(\nu)$$

Where:
- $s_{\text{CMB}}$ = CMB amplitude (this is what we ultimately want!)
- $s_{\text{sync}}$ = synchrotron amplitude at 30 GHz
- $s_{\text{dust}}$ = dust amplitude at 353 GHz
- $f_s(\nu)$ = synchrotron scaling function (we defined this earlier)
- $f_d(\nu)$ = dust scaling function (we defined this earlier)
- $\beta_s$ = synchrotron spectral index (hidden inside $f_s$)

### MCMC on a Single Pixel

Running MCMC on every pixel (128x128 = 16,384 pixels) would take too long. So we'll:

1. **Run MCMC on ONE pixel** to learn the spectral index $\beta_s$
2. **Use that $\beta_s$** to quickly solve all pixels with linear algebra

This hybrid approach gives us the best of both worlds!

In [ ]:
# Extract pixel
cx, cy = data['npix']//2, data['npix']//2
pixel_observed = data['observed_maps'][:, cy, cx]
pixel_noise_std = np.ones(len(FREQUENCIES_GHZ)) * data['noise_std']
true_cmb = data['cmb_input'][cy, cx]
true_sync = data['sync_input'][cy, cx]
true_dust = data['dust_input'][cy, cx]

print(f"Selected pixel: ({cx}, {cy}) - center of the map")
print(f"\nTrue values at this pixel:")
print(f"  CMB amplitude:         {true_cmb:.2f} μK")
print(f"  Synchrotron amplitude: {true_sync:.2f} μK (at 30 GHz)")
print(f"  Dust amplitude:        {true_dust:.2f} μK (at 353 GHz)")
print(f"  Spectral index β_s:    {data['beta_s_true']:.2f}")
print(f"\nObserved data at 7 frequencies: {np.round(pixel_observed, 1)} μK")

---
## Exercise 1: Implement The Forward Model

Complete the function that predicts observations given model parameters.

**What you need to do:**
1. Calculate the CMB contribution: This is constant at all frequencies, so it's just `s_cmb` times an array of ones

2. Calculate the synchrotron contribution: Multiply `s_sync` by the synchrotron scaling function

3. Calculate the dust contribution: Multiply `s_dust` by the dust scaling function

4. Add them all together to get the total model prediction

In [ ]:
def forward_model(params, frequencies_hz):
    """
    Predict what we would observe given model parameters.

    This is called a "forward model" because it goes forward from
    parameters to predictions (the opposite of inference, which goes
    from data back to parameters).

    Parameters:
        params: Dictionary with keys 's_cmb', 's_sync', 's_dust', 'beta_s'
        frequencies_hz: Array of observation frequencies in Hz

    Returns:
        Array of predicted observations at each frequency (in μK)
    """
    # Extract parameters from the dictionary
    s_cmb = params['s_cmb']     # CMB amplitude
    s_sync = params['s_sync']   # Synchrotron amplitude at 30 GHz
    s_dust = params['s_dust']   # Dust amplitude at 353 GHz
    beta_s = params['beta_s']   # Synchrotron spectral index

    # ============================================================
    # TODO: Calculate the model prediction at each frequency
    #
    # Hint: The total signal is:
    #   model = CMB + synchrotron + dust
    #
    # Where:
    #   CMB contribution = s_cmb * np.ones(len(frequencies_hz))
    #   Synchrotron contribution = s_sync * synchrotron_spectrum(frequencies_hz, beta_s)
    #   Dust contribution = s_dust * dust_spectrum(frequencies_hz)
    # ============================================================

    model = None  # <-- Replace this with your calculation!

    return model

<details>
<summary>💡 Click here for the solution</summary>

```python
# CMB: constant at all frequencies
cmb = s_cmb * np.ones(len(frequencies_hz))

# Synchrotron: amplitude × scaling function
sync = s_sync * synchrotron_spectrum(frequencies_hz, beta_s)

# Dust: amplitude × scaling function
dust = s_dust * dust_spectrum(frequencies_hz)

# Total model is the sum
model = cmb + sync + dust
```
</details>

In [ ]:
# Test your forward model
test_params = {
    's_cmb': true_cmb,
    's_sync': true_sync,
    's_dust': true_dust,
    'beta_s': data['beta_s_true']
}

model_prediction = forward_model(test_params, FREQUENCIES_HZ)

if model_prediction is not None:
    print("✓ Forward model test:")
    print(f"  Model prediction: {np.round(model_prediction, 1)} μK")
    print(f"  Observed data:    {np.round(pixel_observed, 1)} μK")
    print(f"  Difference:       {np.round(pixel_observed - model_prediction, 1)} μK")
    print("\n  The difference is just noise - the model is working!")
else:
    print("❌ Please complete the forward_model function above!")

---
## Exercise 2: The Likelihood Function

### What is the Likelihood?

The **likelihood** tells us how well our model fits the data:

- If model predictions are **close** to observations → **high** likelihood (good!)
- If model predictions are **far** from observations → **low** likelihood (bad!)

### The Math

We assume the noise follows a **Gaussian** (bell curve) distribution. The log-likelihood is:

$$\log L = -\frac{1}{2} \sum_{i=1}^{7} \left(\frac{\text{data}_i - \text{model}_i}{\sigma_i}\right)^2$$

This is basically $-\chi^2/2$. The closer the model is to the data, the larger (less negative) this number.

### Why Log?

We use the **logarithm** of the likelihood because:
1. Likelihoods can be tiny numbers (like 10⁻¹⁰⁰) that cause computer errors
2. Log turns multiplication into addition (easier math)
3. The maximum of log(L) is the same as the maximum of L

In [ ]:
def log_likelihood(params, observed, noise_std, frequencies_hz):
    """
    Calculate how well the model fits the data.

    Returns the log of the likelihood (we use log because the actual
    likelihood can be a tiny number that causes computer errors).

    Parameters:
        params: Dictionary of model parameters
        observed: Array of observed data at each frequency
        noise_std: Array of noise standard deviations
        frequencies_hz: Array of frequencies

    Returns:
        Log-likelihood value (higher = better fit)
    """
    # Get model prediction
    model = forward_model(params, frequencies_hz)

    if model is None:
        return -np.inf  # Return negative infinity if model failed

    # ============================================================
    # TODO: Calculate the log-likelihood
    #
    # Step 1: Calculate residuals (difference between data and model)
    #         residuals = observed - model
    #
    # Step 2: Calculate chi-squared
    #         chi_squared = sum of (residuals / noise_std)^2
    #
    # Step 3: Log-likelihood = -0.5 * chi_squared
    # ============================================================
    log_L = None  # <-- Replace this!

    return log_L

<details>
<summary>💡 Click here for the solution</summary>

```python
# Step 1: Residuals
residuals = observed - model

# Step 2: Chi-squared (sum of squared residuals, weighted by noise)
chi_squared = np.sum((residuals / noise_std) ** 2)

# Step 3: Log-likelihood
log_L = -0.5 * chi_squared
```
</details>

In [ ]:
# Test the likelihood function
if model_prediction is not None:
    # Likelihood with true parameters
    log_L_true = log_likelihood(test_params, pixel_observed, pixel_noise_std, FREQUENCIES_HZ)

    # Likelihood with wrong parameters
    wrong_params = {'s_cmb': 50.0, 's_sync': 100.0, 's_dust': 10.0, 'beta_s': -2.0}
    log_L_wrong = log_likelihood(wrong_params, pixel_observed, pixel_noise_std, FREQUENCIES_HZ)

    if log_L_true is not None:
        print("✓ Likelihood test:")
        print(f"  True parameters:  log L = {log_L_true:.2f}")
        print(f"  Wrong parameters: log L = {log_L_wrong:.2f}")
        print(f"\n  Higher is better! True parameters fit better.")

---
## Exercise 3: The Prior Distribution

### What is the Prior?

The **prior** encodes what we know about the parameters **before** seeing the data. It represents our physical knowledge and constraints.

### Why Do We Need Priors?

1. **Physical constraints**: Some values are impossible (e.g., negative emission intensity)
2. **Prior knowledge**: We know from physics that $\beta_s$ is around -3
3. **Regularization**: Priors prevent crazy parameter values

### Our Priors

| Parameter | Prior Type | Range/Values | Reason |
|-----------|------------|--------------|--------|
| $s_{\text{CMB}}$ | Uniform | -500 to +500 μK | Temperature can be + or - |
| $s_{\text{sync}}$ | Uniform | 0 to 500 μK | Emission is always positive |
| $s_{\text{dust}}$ | Uniform | 0 to 500 μK | Emission is always positive |
| $\beta_s$ | Gaussian | Mean=-3, σ=0.5 | Physics tells us it's around -3 |

### Log-Prior

- If parameter is **outside** allowed range → return $-\infty$ (probability = 0)
- Otherwise → return log of prior probability

In [ ]:
from logging import log
def log_prior(params):
    """
    Calculate the prior probability of the parameters.

    Returns -infinity if parameters are outside allowed range
    (meaning probability = 0, these values are impossible).

    Parameters:
        params: Dictionary of model parameters

    Returns:
        Log of prior probability
    """
    s_cmb = params['s_cmb']
    s_sync = params['s_sync']
    s_dust = params['s_dust']
    beta_s = params['beta_s']

    # ============================================================
    # TODO: Implement the prior
    #
    # Step 1: Check if parameters are in allowed range
    #         If not, return -np.inf (probability = 0)
    #
    #   Allowed ranges:
    #   - s_cmb: between -500 and 500
    #   - s_sync: between 0 and 500 (must be positive!)
    #   - s_dust: between 0 and 500 (must be positive!)
    #   - beta_s: between -5 and -1
    #
    # Step 2: Add informative prior on beta_s
    #         Gaussian centered on -3.0 with width 0.5:
    #         log_p_beta = -0.5 * ((beta_s - (-3.0)) / 0.5)^2
    # ============================================================

    log_p = None  # <-- Replace this!

    return log_p

<details>
<summary>💡 Click here for the solution</summary>

```python
# Check bounds - return -inf if outside (probability = 0)
if not (-500 < s_cmb < 500):
    return -np.inf
if not (0 < s_sync < 500):
    return -np.inf
if not (0 < s_dust < 500):
    return -np.inf
if not (-5 < beta_s < -1):
    return -np.inf

# Gaussian prior on beta_s (we know it should be around -3)
log_p_beta = -0.5 * ((beta_s - (-3.0)) / 0.5) ** 2

# Uniform priors on other parameters contribute 0 to log-prior
log_p = log_p_beta
```
</details>

In [ ]:
# The posterior combines likelihood and prior
def log_posterior(params, observed, noise_std, frequencies_hz):
    """
    The posterior probability is proportional to likelihood × prior.

    In log space: log(posterior) = log(likelihood) + log(prior)

    This is Bayes' theorem in action!
    """
    # First check the prior
    lp = log_prior(params)

    # If prior is -infinity, don't bother computing likelihood
    if not np.isfinite(lp):
        return -np.inf

    # Add log-likelihood to log-prior
    ll = log_likelihood(params, observed, noise_std, frequencies_hz)

    return lp + ll

print("✓ Posterior function defined!")

---
# Part 2: MCMC Sampling

Now comes the exciting part: **Markov Chain Monte Carlo (MCMC)**!

## What is MCMC?

MCMC is a clever way to explore the space of possible parameter values and find where the posterior probability is highest.

### Analogy: Exploring a Mountain Range Blindfolded

Imagine you're blindfolded in a hilly landscape and want to find the highest peak:

1. You're standing at some position
2. Take a random step in some direction
3. If you went **uphill** → definitely stay at the new position
4. If you went **downhill** → maybe stay anyway (with some probability)
5. Repeat thousands of times

Over time, you'll spend most of your time near the highest peaks! The collection of all positions you visited approximates the shape of the probability distribution.

## The Metropolis-Hastings Algorithm

Here's the algorithm step by step:

```
1. START at initial parameter values θ
2. FOR each step:
   a. PROPOSE: Add random noise to get a new point θ'
      θ' = θ + random_noise
   
   b. EVALUATE: Calculate posterior probability at new point P(θ')
   
   c. DECIDE: Accept or reject the proposal
      - If P(θ') > P(θ): always accept (moving uphill)
      - If P(θ') < P(θ): accept with probability P(θ')/P(θ) (sometimes accept downhill)
   
   d. RECORD: Store the current position
   
3. The collection of stored positions = samples from the posterior!
```

## Exercise 4: Implement the MCMC Sampler

This is the main algorithm! Fill in the missing pieces.

In [ ]:
def metropolis_hastings(initial_params, n_steps, proposal_std,
                        observed, noise_std, frequencies_hz):
    """
    Run the Metropolis-Hastings MCMC algorithm.

    Parameters:
        initial_params: Starting values for the parameters
        n_steps: Number of MCMC steps to take
        proposal_std: How far to jump when proposing new points
        observed: The observed data
        noise_std: Noise level in the data
        frequencies_hz: Observation frequencies

    Returns:
        chain: Dictionary with arrays of parameter values at each step
        acceptance_rate: Fraction of proposals that were accepted
    """
    # Parameter names
    param_names = ['s_cmb', 's_sync', 's_dust', 'beta_s']

    # Initialize storage for the chain
    # We'll store the parameter values at each step
    chain = {name: np.zeros(n_steps) for name in param_names}

    # Start at the initial parameters
    current_params = initial_params.copy()
    current_log_post = log_posterior(current_params, observed, noise_std, frequencies_hz)

    # Count how many proposals we accept
    n_accepted = 0

    # Main MCMC loop
    for i in tqdm(range(n_steps), desc="MCMC Sampling"):

        # ============================================================
        # TODO: Implement one MCMC step
        #
        # Step 1: PROPOSE a new point
        #         For each parameter, add Gaussian noise:
        #         proposed_params[name] = current_params[name] + random_noise
        #         where random_noise = np.random.normal(0, proposal_std[name])
        #
        # Step 2: EVALUATE the proposed point
        #         proposed_log_post = log_posterior(proposed_params, ...)
        #
        # Step 3: DECIDE whether to accept
        #         log_alpha = proposed_log_post - current_log_post
        #         if np.log(np.random.random()) < log_alpha:
        #             accept! (update current_params and current_log_post)
        #             n_accepted += 1
        # ============================================================

        pass  # <-- Replace this with your implementation!

        # Store current state in the chain
        for name in param_names:
            chain[name][i] = current_params[name]

    # Calculate and report acceptance rate
    acceptance_rate = n_accepted / n_steps
    print(f"\n✓ MCMC complete!")
    print(f"  Acceptance rate: {acceptance_rate:.1%}")
    print(f"  (Ideal range: 20% - 50%)")

    return chain, acceptance_rate

<details>
<summary>💡 Click here for the solution</summary>

```python
# Step 1: PROPOSE a new point
proposed_params = {}
for name in param_names:
    # Add Gaussian noise to current value
    proposed_params[name] = current_params[name] + np.random.normal(0, proposal_std[name])

# Step 2: EVALUATE the proposed point
proposed_log_post = log_posterior(proposed_params, observed, noise_std, frequencies_hz)

# Step 3: DECIDE whether to accept
# log_alpha is the log of the acceptance probability
log_alpha = proposed_log_post - current_log_post

# Accept if a random number is less than the acceptance probability
# (using logs to avoid numerical issues)
if np.log(np.random.random()) < log_alpha:
    # Accept! Move to the new point
    current_params = proposed_params
    current_log_post = proposed_log_post
    n_accepted += 1
# If we don't accept, we just stay at current_params (do nothing)
```
</details>

In [ ]:
# Run the MCMC!

# Starting point (deliberately not the true values)
initial_params = {
    's_cmb': 0.0,      # Start at 0, let MCMC find the true value
    's_sync': 50.0,    # Reasonable guess
    's_dust': 50.0,    # Reasonable guess
    'beta_s': -2.8     # Close to -3 but not exactly
}

# Proposal widths - how far to jump each step
# Too small = slow exploration, too large = many rejections
proposal_std = {
    's_cmb': 8.0,
    's_sync': 8.0,
    's_dust': 5.0,
    'beta_s': 0.15
}

# Run for 10,000 steps
n_steps = 10000

print("="*60)
print("RUNNING MCMC")
print("="*60)
print(f"Starting point:")
print(f"  s_cmb = {initial_params['s_cmb']}, s_sync = {initial_params['s_sync']}")
print(f"  s_dust = {initial_params['s_dust']}, beta_s = {initial_params['beta_s']}")
print(f"Number of steps: {n_steps:,}")
print()

# Run MCMC!
chain, acceptance_rate = metropolis_hastings(
    initial_params, n_steps, proposal_std,
    pixel_observed, pixel_noise_std, FREQUENCIES_HZ
)

---
## Part 3: Analyzing the Results

The MCMC has given us a "chain" of parameter values. Let's visualize what happened!

### Trace Plots

Trace plots show how each parameter evolved during the MCMC run. We want to see:
- The chain finds the true value (red dashed line)
- The chain "mixes" well (bounces around, doesn't get stuck)

In [ ]:
# Create trace plots showing how parameters evolved during MCMC
param_names = ['s_cmb', 's_sync', 's_dust', 'beta_s']
param_labels = ['CMB [μK]', 'Synchrotron [μK]', 'Dust [μK]', 'β_s']
true_values = [true_cmb, true_sync, true_dust, data['beta_s_true']]

fig, axes = plt.subplots(4, 1, figsize=(12, 10))

for ax, name, label, true_val in zip(axes, param_names, param_labels, true_values):
    # Plot the chain (parameter value at each step)
    ax.plot(chain[name], 'b-', alpha=0.5, lw=0.5)

    # Mark the true value with a red dashed line
    ax.axhline(true_val, color='red', ls='--', lw=2, label=f'True = {true_val:.2f}')

    ax.set_ylabel(label, fontsize=11)
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('MCMC Step', fontsize=12)
axes[0].set_title('Trace Plots: How Parameters Evolved During MCMC', fontsize=14)
plt.tight_layout()
plt.show()

print("\n📊 WHAT TO LOOK FOR:")
print("   • Each chain should fluctuate around the true value (red line)")
print("   • The first ~1000 steps may show 'burn-in' (finding the right region)")
print("   • Good mixing = lots of up-and-down movement (not stuck)")

### Posterior Distributions

By throwing away the "burn-in" period (where the chain was still finding the right region) and making histograms of the remaining samples, we get the **posterior distribution** — our estimate of each parameter with uncertainties!

In [ ]:
# Discard burn-in (first 2000 steps while chain was finding the right region)
burn_in = 2000

# Create new dictionary with only post-burn-in samples
samples = {}
for name in param_names:
    samples[name] = chain[name][burn_in:]  # [burn_in:] means "from burn_in to the end"

print(f"Discarding first {burn_in} samples as burn-in")
print(f"Using remaining {len(samples['s_cmb']):,} samples for analysis")

In [ ]:
# Plot posterior distributions
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for ax, name, label, true_val in zip(axes, param_names, param_labels, true_values):
    # Histogram of samples = posterior distribution
    ax.hist(samples[name], bins=50, density=True, alpha=0.7,
            color='steelblue', edgecolor='white')

    # Mark true value
    ax.axvline(true_val, color='red', ls='--', lw=2,
               label=f'True = {true_val:.2f}')

    # Mark posterior mean and std
    mean = np.mean(samples[name])
    std = np.std(samples[name])
    ax.axvline(mean, color='black', lw=2,
               label=f'Mean = {mean:.2f} ± {std:.2f}')

    ax.set_xlabel(label, fontsize=11)
    ax.set_ylabel('Probability Density', fontsize=11)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('Posterior Distributions: Our Parameter Estimates with Uncertainties',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Print summary table
print("="*65)
print("PARAMETER ESTIMATES FROM MCMC")
print("="*65)
print(f"{'Parameter':<20} {'True Value':<12} {'Estimate':<20} {'Error':<10}")
print("-"*65)

for name, label, true_val in zip(param_names,
                                  ['CMB amp', 'Sync amp', 'Dust amp', 'Beta_s'],
                                  true_values):
    mean = np.mean(samples[name])
    std = np.std(samples[name])
    error = mean - true_val

    print(f"{label:<20} {true_val:<12.2f} {mean:.2f} ± {std:.2f}        {error:+.2f}")

print("="*65)
print("\n✓ We successfully recovered the parameters!")
print("✓ The uncertainties (±) tell us how confident we are.")

---
## Full Map Reconstruction

Now we'll use what we learned from MCMC to recover the CMB across the entire map.

### The Strategy

1. **From MCMC**: We learned the spectral index $\beta_s$ (and its uncertainty)
2. **For each pixel**: Use linear algebra to find the best-fit amplitudes

### Why This Works

Once we know $\beta_s$, the model becomes **linear** in the amplitudes:

$$\begin{pmatrix} d_{30} \\ d_{44} \\ \vdots \\ d_{353} \end{pmatrix} = \begin{pmatrix} 1 & f_s(30) & f_d(30) \\ 1 & f_s(44) & f_d(44) \\ \vdots & \vdots & \vdots \\ 1 & f_s(353) & f_d(353) \end{pmatrix} \begin{pmatrix} s_{\text{CMB}} \\ s_{\text{sync}} \\ s_{\text{dust}} \end{pmatrix}$$

This is just: **data = mixing_matrix × amplitudes**

We can solve this with least squares, which is very fast!

In [ ]:
def reconstruct_cmb_map(observed_maps, beta_s_estimate, frequencies_hz):
    """
    Reconstruct the CMB map using the spectral index from MCMC.

    For each pixel, we solve: data = A @ amplitudes
    where A is the mixing matrix and amplitudes = [s_cmb, s_sync, s_dust]

    Parameters:
        observed_maps: Array of shape (n_freq, ny, nx)
        beta_s_estimate: Spectral index from MCMC
        frequencies_hz: Observation frequencies

    Returns:
        cmb_map: Recovered CMB map
    """
    n_freq, ny, nx = observed_maps.shape
    A = np.column_stack([np.ones(n_freq), synchrotron_spectrum(frequencies_hz, beta_s_est), dust_spectrum(frequencies_hz)])
    cmb_map = np.zeros((ny, nx))
    for j in range(ny):
        for i in range(nx):
            amps, _, _, _ = np.linalg.lstsq(A, observed_maps[:, j, i], rcond=None)
            cmb_map[j, i] = amps[0]
    return cmb_map

    print("Done!")
    return cmb_map

In [ ]:
# Compare maps: True vs Recovered vs Residual
beta_s_est = np.mean(samples['beta_s'])
print(f"Using β_s = {beta_s_est:.3f}")
cmb_recovered = reconstruct_cmb_map(data['observed_maps'], beta_s_est, FREQUENCIES_HZ)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
cmb_true = data['cmb_input']
residual = cmb_recovered - cmb_true
vmax = np.percentile(np.abs(cmb_true), 98)
vmax_res = np.percentile(np.abs(residual), 98)

im0 = axes[0].imshow(cmb_true, cmap='RdBu_r', origin='lower', vmin=-vmax, vmax=vmax)
axes[0].set_title('True CMB (Input)')
plt.colorbar(im0, ax=axes[0], label='μK')

im1 = axes[1].imshow(cmb_recovered, cmap='RdBu_r', origin='lower', vmin=-vmax, vmax=vmax)
axes[1].set_title('Recovered CMB')
plt.colorbar(im1, ax=axes[1], label='μK')

im2 = axes[2].imshow(residual, cmap='RdBu_r', origin='lower', vmin=-vmax_res, vmax=vmax_res)
axes[2].set_title(f'Residual (RMS: {np.std(residual):.2f} μK)')
plt.colorbar(im2, ax=axes[2], label='μK')

plt.tight_layout()
plt.show()

### Comparing Our Result to the Truth

In [ ]:
# Quantitative comparison
correlation = np.corrcoef(cmb_true.flatten(), cmb_recovered.flatten())[0, 1]
variance_explained = 1 - np.var(residual) / np.var(cmb_true)

print("="*55)
print("CMB RECOVERY RESULTS")
print("="*55)
print(f"  Input CMB RMS:       {np.std(cmb_true):.2f} μK")
print(f"  Recovered CMB RMS:   {np.std(cmb_recovered):.2f} μK")
print(f"  Residual RMS:        {np.std(residual):.2f} μK")
print(f"")
print(f"  Correlation:         {correlation:.4f}  (1.0 = perfect)")
print(f"  Variance explained:  {variance_explained*100:.1f}%")
print("="*55)

if correlation > 0.99:
    print("\n🎉 Excellent! We recovered the CMB very accurately!")
elif correlation > 0.95:
    print("\n✓ Good recovery! Some residual contamination remains.")
else:
    print("\n⚠ Something might be wrong - check the exercises above.")

In [ ]:
# Scatter plot: pixel-by-pixel comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: scatter plot
axes[0].scatter(cmb_true.flatten(), cmb_recovered.flatten(), alpha=0.1, s=2, c='blue')
lims = [-np.percentile(np.abs(cmb_true), 99), np.percentile(np.abs(cmb_true), 99)]
axes[0].plot(lims, lims, 'r--', lw=2, label='Perfect recovery')
axes[0].set_xlabel('True CMB [μK]', fontsize=12)
axes[0].set_ylabel('Recovered CMB [μK]', fontsize=12)
axes[0].set_title(f'Pixel-by-Pixel Comparison\n(correlation = {correlation:.4f})', fontsize=13)
axes[0].legend()
axes[0].set_aspect('equal')
axes[0].grid(True, alpha=0.3)

# Right: residual histogram
axes[1].hist(residual.flatten(), bins=60, density=True, alpha=0.7,
             color='steelblue', edgecolor='white')
axes[1].axvline(0, color='red', ls='--', lw=2)
axes[1].set_xlabel('Residual [μK]', fontsize=12)
axes[1].set_ylabel('Probability Density', fontsize=12)
axes[1].set_title(f'Residual Distribution\n(should be centered at 0)', fontsize=13)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Summary: What We Accomplished

## The Problem
We wanted to extract the CMB signal from observations contaminated by galactic foregrounds (synchrotron and dust).

## Our Approach: MCMC + Linear Algebra

1. **Built a parametric model** that predicts observations given component amplitudes and spectral parameters

2. **Used MCMC on one pixel** to learn the synchrotron spectral index $\beta_s$ (and got uncertainties for free!)

3. **Applied linear algebra** to quickly reconstruct the full CMB map using the learned $\beta_s$

## Key Concepts We Learned

| Concept | What It Means | Why It Matters |
|---------|---------------|----------------|
| **Forward model** | Predicts data from parameters | Lets us compare model to observations |
| **Likelihood** | How well model fits data | Higher = better fit |
| **Prior** | What we knew before seeing data | Encodes physical constraints |
| **Posterior** | Updated knowledge after seeing data | Combines likelihood and prior |
| **MCMC** | Algorithm to sample the posterior | Explores parameter space efficiently |

## The Power of Bayesian Methods

Unlike simpler methods, MCMC gives us **uncertainties** on our estimates. We don't just know the CMB amplitude — we know how confident we are in that value!

## Real-World Applications

This is a simplified version of what real CMB experiments do:
- **Planck's Commander** uses similar Bayesian methods
- **BeyondPlanck** is a modern implementation
- These methods enabled the discovery of the CMB's primordial fluctuation patterns!

---

# 🎉 Congratulations!

You've successfully implemented Bayesian CMB foreground separation using Monte Carlo methods.